# Xenium Human Breast Cancer Sample Integration with Human PPI Gene Programs

- **Creator**: Sebastian Birk (<sb75@sanger.ac.uk>).
- **Affiliation:** Wellcome Sanger Institute
- **Date of Creation:** 26.08.2026
- **Date of Last Modification:** 26.08.2026

In this tutorial we apply NicheCompass to integrate two replicates of the 10x Xenium human breast cancer
dataset from [Janesick, A. et al. High resolution mapping of the tumour microenvironment using integrated
single-cell, spatial and in situ analysis. Nat. Commun. 14, 8353 (2023)](https://www.nature.com/articles/s41467-023-43458-x),
as analyzed in the NicheCompass paper ([Birk, S. et al. Quantitative characterization of cell niches in
spatially resolved omics data. Nat. Genet. 57, 897-909 (2025)](https://www.nature.com/articles/s41588-025-02120-6)).

In contrast to the other tutorials, the prior knowledge gene program (GP) mask is built **exclusively** from
predicted human protein-protein interactions (PPIs) taken from the human interactome of
[Zhang, J., Humphreys, I. R. et al. Predicting protein-protein interactions in the human proteome.
Science 390, eadt1630 (2025)](https://www.science.org/doi/10.1126/science.adt1630), instead of from
OmniPath, NicheNet and MEBOCOST.

Sample 1 (batch1) and Sample 2 (batch2) each have:
- observations at cellular resolution with cell state annotations (```cell_states```)
- 313 probed genes (Xenium Human Breast panel: 280 base genes + 33 custom add-on genes)

- Check the [documentation](https://nichecompass.readthedocs.io/en/latest/installation.html) for NicheCompass installation instructions.
- This notebook requires a NicheCompass version that contains ```extract_gp_dict_from_humanppi_interactions```
  (available from the ```feature/zhang_ppi_gps``` development line). If you installed NicheCompass from PyPI,
  install the local checkout instead: ```pip install -e <path_to_nichecompass>```.
- The data for this notebook is expected under ```<repository_root>/datasets/st_data/xenium_human_breast_cancer/```:
  - xenium_human_breast_cancer_batch1.h5ad
  - xenium_human_breast_cancer_batch2.h5ad
- The human PPI predictions are downloaded automatically on first use (~14 MB) and cached under
  ```<repository_root>/datasets/gp_data/```. The download server presents an incomplete TLS certificate
  chain, so you will see a one-off ```UserWarning``` about certificate verification being disabled.
  The data is also archived on [Dryad](https://doi.org/10.5061/dryad.15dv41p84).

> **Important caveat for this dataset.** The Xenium breast panel probes only 313 genes, whereas an
> intercellular PPI GP consists of exactly one source gene and one target gene, and both must be probed
> for the GP to represent a real interaction. Out of 5,351 interactions classified as intercellular at 80%
> precision, only **42** have both partners on this panel. This notebook therefore produces a small GP mask
> (tens, not thousands, of prior GPs), which is expected: the PPI resource is best suited to
> whole-transcriptome data, and on a targeted panel it is more informative as a *supplement* to
> OmniPath/NicheNet/MEBOCOST than on its own. We keep it standalone here to isolate and inspect its
> contribution. Because the prior mask is small, the de novo GPs (```n_addon_gp```) will carry a
> comparatively large share of the embedding.

## 1. Setup

### 1.1 Import Libraries

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import random
import warnings
from datetime import datetime

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns
import squidpy as sq
from matplotlib import gridspec
from sklearn.preprocessing import MinMaxScaler

from nichecompass.models import NicheCompass
from nichecompass.utils import (add_gps_from_gp_dict_to_adata,
                                create_new_color_dict,
                                compute_communication_gp_network,
                                visualize_communication_gp_network,
                                extract_gp_dict_from_humanppi_interactions,
                                filter_and_combine_gp_dict_gps_v2,
                                generate_enriched_gp_info_plots)

### 1.2 Define Parameters

Most parameters are taken from the other NicheCompass tutorials and from the published Xenium human
breast cancer reference run. The parameters that are **specific to the human PPI gene program (GP) mask**
are new and are documented here in full.

All GP counts below are **resource-level numbers for ```species = "human"```**: they are what
```extract_gp_dict_from_humanppi_interactions``` returns, independent of any dataset, gene panel or masking
threshold. How many survive masking against the 313 probed genes of *this* dataset is treated separately at
the end of this section.

### How interactions are classified

An interaction can only act between cells if **both** partners present a face to the outside of the cell.
The extractor therefore classifies each protein first, and each interaction second.

**Step 1 — location class of each protein**, from its UniProt cellular-component keywords:

| Location class | Meaning | Example keywords |
| :-- | :-- | :-- |
| `cell_surface` | membrane anchored with a necessarily extracellular face | `Cell membrane`, `Cell surface`, `Apical/Basolateral cell membrane`, `Sarcolemma`, `Gap junction`, `MHC I`, `MHC II`, `T cell receptor`, `Target cell membrane` |
| `secreted` | released into the extracellular space, not membrane anchored | `Secreted`, `Extracellular matrix`, `Basement membrane`, `Membrane attack complex`, `HDL`/`LDL`/`VLDL`/`Chylomicron`, `Surface film` |
| `intracellular` | an intracellular location, and no surface or secreted keyword | `Nucleus`, `Cytoplasm`, `Mitochondrion`, `Endoplasmic reticulum`, `Golgi apparatus`, `Lysosome`, `Endosome`, `Cytoskeleton`, `Proteasome`, `Spliceosome`, `Coated pit`, `Synaptosome`, `Flagellum`, `Exosome` |
| `ambiguous` | compatible with an extracellular face but does not establish one | `Membrane`, `Cell junction`, `Tight junction`, `Adherens junction`, `Desmosome`, `Focal adhesion`, `Cell projection`, `Cilium`, `Synapse`, `Microvillus`, `Filopodium`, `Amyloid`, `Immunoglobulin`, `Virion` |
| `unknown` | no usable keyword at all | — |

Two precedence rules apply, in this order:

1. **An extracellular face beats an intracellular location.** Secreted and surface proteins are routinely
   also annotated with the compartments they traverse — interleukin 15 is `Cytoplasm,Nucleus,Secreted` — so
   requiring the absence of intracellular keywords would discard genuine ligands.
2. **Membrane anchoring beats secretion.** 345 proteins carry both, because many surface receptors have a
   shed soluble isoform — PD-L1 is `Cell membrane,Endosome,Membrane,Nucleus,Secreted`. Calling those
   `secreted` would turn contact-dependent interactions such as PD-1 / PD-L1 into paracrine ones.

Why several plausible-sounding keywords are **not** treated as surface:

- `Membrane` is the generic parent of the entire membrane branch and also covers the endoplasmic reticulum,
  mitochondrial, Golgi, nuclear and endolysosomal membranes. UniProt applies the specific child
  `Cell membrane` when plasma-membrane localization is known.
- `Cell junction`, `Tight junction`, `Synapse`, `Cell projection` and `Cilium` denote compartments that
  consist of a membrane-embedded core **and** a large cytoplasmic component. They are carried by cytosolic
  plaque and scaffold proteins (ZO-1, catenins, vinculin, talin, paxillin, PSD-95, synapsins) and by
  axonemal and intraflagellar-transport machinery. The genuinely surface-exposed proteins in those
  compartments virtually always also carry `Cell membrane`, so excluding these keywords costs little
  coverage and removes a large false-positive source.
- `Exosome` in UniProt denotes the **exosome complex**, the nuclear/cytoplasmic 3'-5' exoribonuclease
  machine (EXOSC subunits), not extracellular vesicles.
- `Amyloid` describes an aggregation propensity rather than a location, and `Immunoglobulin` is also used
  for the immunoglobulin *domain*, which occurs in intracellular proteins such as titin and obscurin.

**Step 2 — interaction class**, from the two location classes:

| Interaction class | Condition | Interpretation |
| :-- | :-- | :-- |
| `juxtacrine` | both partners extracellular facing, neither purely secreted, **and both able to reach across the intercellular cleft** | contact-dependent signalling between touching cells |
| `paracrine` | both partners extracellular facing, at least one secreted | signalling through a diffusible partner |
| `cis_complex` | subunits of a common protein complex, a curated co-receptor family, or a partner that cannot reach across the cleft | assembles within one cell, so not signalling between cells |
| `extracellular_assembly` | two **secreted** subunits of a common complex | a multimer assembled by the cell that secretes it |
| `intracellular` | at least one partner has no extracellular face | a within-cell complex |
| `unknown` | at least one partner could not be localized | unclassifiable |

Three mechanisms separate same-cell complexes from genuine signalling between cells, because localization
alone cannot: two subunits of a channel or a receptor heterodimer both face outwards and neither is
secreted, which satisfies `juxtacrine` on localization evidence alone.

1. **Protein complex membership** (`humanppi_detect_cis_complexes`), from the EBI Complex Portal plus 36
   curated gene families (MHC class I and II, the T cell receptor chains, CD8, CD79, the IgE receptor,
   CD94/NKG2, integrins, collagens, laminins, and 12 co-receptor families covering the ERBB heterodimers,
   the insulin receptors, the TGF-beta receptors including endoglin, the Toll-like receptor heterodimers,
   the metabotropic GABA receptor, the Hedgehog receptor, the neuropilins with the VEGF receptors, the
   calcitonin receptors, the acid-sensing channels, the aquaporins and the sterol and organic solute
   transporters). The Complex Portal covers only about a fifth of the proteins in the resource and misses
   several prominent surface complexes outright, hence the curated families. Adjudication is at pair level:
   a shared complex that pairs a secreted with a membrane-anchored partner is a genuine ligand-receptor
   assembly and stays paracrine. **The Complex Portal is used only to reclassify interactions, never to add
   any.**
2. **A positive test for reach** (`humanppi_min_extracellular_domain_length`, default 30 residues). The
   length of each annotated extracellular topological domain is parsed from UniProt, and a
   contact-dependent interaction is reclassified `cis_complex` when either partner's longest extracellular
   domain falls below the threshold, since a protein that barely protrudes from its own membrane cannot
   bridge the cleft. `CD247` has a 9-residue ectodomain, `FCER1G` 5, `KCNJ6` 24. Only positive evidence
   demotes, so an unannotated protein is not penalized, and connexins, claudins and occludin are exempt
   because their docking through short loops is genuine. The test is **not** applied to paracrine
   interactions, where the soluble partner diffuses and can engage a shallow pocket — `CCR7` has only a
   35-residue extracellular N-terminus.
3. **Membrane topology** (`humanppi_use_topology`), which decides whether a `Cell membrane` keyword really
   implies an extracellular face. Peripheral proteins docked onto the cytoplasmic leaflet carry that keyword
   too; without topology the SNARE complex, the protein kinase A holoenzyme and the cytoplasmic adherens
   plaque are all called contact-dependent signalling.

Together these reclassify 210 of 1,073 candidate contact-dependent interactions at precision `"90"`. All 15
canonical same-cell pairs probed during validation are now non-intercellular, and recall against curated
OmniPath ligand-receptor pairs is 97.0% (392 of 404).

`intercellular` in `humanppi_program_type` means `paracrine` + `juxtacrine`.

### `humanppi_precision`
| Option | Gene programs (`both`, defaults) | Comment |
| :-- | --: | :-- |
| `"90"` | 16,847 | Expected precision 90%. Released table: 17,849 rows. |
| `"80"` | 27,823 | Expected precision 80%. Superset of `"90"`. Released table: 29,257 rows. |

GP counts fall below the row counts because a few interactions are listed twice, because interactions
involving a protein with no UniProt gene name are dropped, because immunoglobulin and T cell receptor gene
segments and non-heterodimerizing paralogue pairs are filtered, and because unclassifiable interactions are
excluded by default (see `humanppi_unresolved_locality`).

### `humanppi_program_type`

| Option | precision `"90"` | precision `"80"` | GP shape |
| :-- | --: | --: | :-- |
| `"intercellular"` (used here) | 1,584 | 3,088 | source = ligand, target = receptor (see `humanppi_orient_juxtacrine_gps`) |
| `"intracellular"` | 15,263 | 24,735 | empty source, both partners in the target |
| `"both"` | 16,847 | 27,823 | each interaction routed to the appropriate shape |

The `both` row is the sum of the other two: `program_type` only selects which classes are kept. Split by
interaction class:

| Class | precision `"90"` | precision `"80"` |
| :-- | --: | --: |
| `paracrine` | 815 | 1,424 |
| `juxtacrine` | 769 | 1,664 |
| `cis_complex` | 551 | 1,224 |
| `extracellular_assembly` | 98 | 202 |
| `intracellular` | 14,614 | 23,309 |

Antibody chains are treated as `secreted` even though they also carry `Cell membrane`, because the secreted
antibody form dominates their interactions with Fc receptors; without that exception, canonical pairs such
as IgG1 / Fc-gamma-receptor-IIb would be called contact dependent.

> **Interaction with section 2.3.** Intracellular GPs have an *empty* source component. Section 2.3 requires
> ```min_source_genes_per_gp=1```, so with `"intracellular"` or `"both"` every intracellular GP is discarded
> at the masking step. To keep them, set ```min_source_genes_per_gp=0``` there.

### `humanppi_ambiguous_locality`

Decides whether a protein whose extracellular face is only *compatible* with its keywords, never
established, counts as able to act between cells.

| Option | Effect |
| :-- | :-- |
| `"extracellular"` (default) | such proteins count as extracellular facing |
| `"intracellular"` | they do not |

**With `humanppi_use_topology = True` this parameter has no effect at all**, because topology resolves every
such protein either way, so no protein is ever left in the `ambiguous` class. It becomes the deciding
parameter only with `humanppi_use_topology = False`, where at precision `"90"` it gives 2,485 intercellular
GPs against 1,745. Both are less trustworthy than the 1,719 obtained with topology, which is why topology is
on by default.

### `humanppi_unresolved_locality`
Around a tenth of the proteins in the resource carry no usable cellular-component keyword. Five fallbacks
are tried in turn — subcellular-location comments, Gene Ontology cellular components, signal peptide plus
membrane anchor, an intracellular-process keyword, and a weak-evidence rule — and what remains cannot be
classified either way. The authors screened these proteins deliberately; such pairs are marked `U` in the
resource's `Source` column.

| Option | Effect (`program_type="both"`), precision `"90"` / `"80"` |
| :-- | :-- |
| `"exclude"` (default) | unclassifiable interactions are dropped: 16,847 / 27,823 GPs |
| `"intracellular"` | they are kept as target-only GPs: 17,371 / 28,662 GPs |

The 524 / 839 difference is the number affected. They can never become intercellular GPs, so this setting
only decides whether they enter the target component. The default is `"exclude"` because treating absence of
evidence as evidence of an intracellular location would silently mislabel any genuine intercellular
interaction among them.

### `humanppi_orient_juxtacrine_gps`
NicheCompass reconstructs the source component from the **neighbours** and the target component from the
**cell itself**, so the ligand belongs in the source component and the receptor in the target component.
For a paracrine interaction the soluble partner settles it. For a contact-dependent interaction both
partners are membrane anchored, and without further evidence the orientation is the order of the two
columns in the released table, which is a coin flip: over the 138 interactions with an unambiguous
canonical sender, the ligand landed in the source component 68 times and in the target component 70 times.

| Option | Effect |
| :-- | :-- |
| `True` (default, used here) | An ordered chain of rules decides: curated ligand/receptor gene families, then HGNC `LG`/`R` symbol stems, then the OmniPath intercell annotation, then the Gene Ontology molecular functions, then the GPI anchor. **918 of 1,584** interactions are oriented from evidence, and of the 215 curated directions the chain decides, **215 are correct** (116 of 116 excluding the ephrins). |
| `False` | The released column order is kept for every interaction. |

The rule that decided each GP is stored under `orientation_rule` in the GP dict, and the counts per rule
are printed by the extractor. Interactions no rule can decide keep the released order and are reported
separately — those are overwhelmingly pairs of poorly characterised proteins where no resource states a
direction, so their interaction class is meaningful but their direction is not.

Note that OmniPath is consulted **only** to decide which of two partners already predicted to interact is
the sender. No interaction is ever added from it.

### `humanppi_symmetric_juxtacrine_gps`
A contact-dependent interaction operates in **both** directions between two neighbouring cells, and unlike a
paracrine interaction it has no principled ligand-to-receptor ordering, so which partner ends up in the
source component is arbitrary.

| Option | Effect |
| :-- | :-- |
| `False` (default, used here) | one GP per interaction, oriented by `humanppi_orient_juxtacrine_gps` where evidence allows |
| `True` | two GPs per contact-dependent interaction, one per orientation: `juxtacrine` GPs go from 769 to 1,538 at precision `"90"` |

`True` is more faithful biologically but each pair of GPs is built from the same two genes, so their latent
dimensions become near-mirror images, inflating the latent space with correlated dimensions and making the
enrichment results in section 4 harder to read. Turn it on when the *direction* of a contact-dependent
interaction is itself the question.

### `humanppi_min_rf_prob` and `humanppi_min_af_prob`
Optional **extra** confidence thresholds on top of the authors' own precision calibration:

| Parameter | Column | Meaning |
| :-- | :-- | :-- |
| `humanppi_min_rf_prob` | `RFprob` | **RoseTTAFold2-PPI** probability — the fast coevolution-driven network that screened all ~190 M candidate pairs. |
| `humanppi_min_af_prob` | `AFprob` | **AlphaFold2** probability — the slower structural rescoring of the pairs that passed the RF2-PPI stage. |

A *low* score in one column does not mean the interaction is unreliable: pairs entered the final set through
several routes, and the evidence-guided routes used relaxed cutoffs. In the precision-`"90"` table `RFprob`
has a median of 0.678 with only 39.4% of entries ≥ 0.9, while `AFprob` has a median of 0.943. The thresholds
are therefore aggressive — for `program_type="intercellular"`:

| Thresholds | precision `"90"` | precision `"80"` |
| :-- | --: | --: |
| none (default) | 1,584 | 3,088 |
| `min_rf_prob=0.5` | 819 | 1,337 |
| `min_rf_prob=0.9` | 456 | 560 |
| `min_af_prob=0.5` | 1,347 | 2,487 |
| `min_af_prob=0.9` | 931 | 1,227 |
| `min_rf_prob=0.9`, `min_af_prob=0.9` | 217 | 247 |

### `combine_gps`
| Option | Effect |
| :-- | :-- |
| `False` (used here) | One GP per interaction. The interaction class stays in the GP name, and 32 GPs survive masking on this dataset. |
| `True` | GPs sharing a source gene are merged by ```filter_and_combine_gp_dict_gps_v2``` (3,413 → 1,447 GPs). Merged GPs are renamed ```<GENE>_combined_GP```, which **discards the interaction class**, and only 25 survive masking here. |

### How many of these GPs survive on *this* dataset
Masked against the 313 probed genes, under permissive thresholds
(```min_genes_per_gp=1, min_source_genes_per_gp=0, min_target_genes_per_gp=0```; any one gene probed) and
the strict thresholds used in section 2.3 (```min_genes_per_gp=2, min_source_genes_per_gp=1,
min_target_genes_per_gp=1```; the GP still encodes an interaction). No GP combining applied.

| `program_type` | precision | permissive | strict |
| :-- | :-- | --: | --: |
| `intercellular` | `"90"` | 210 | 17 |
| `intercellular` | `"80"` | 402 | **31** |
| `intracellular` | `"90"` | 521 | 0 |
| `intracellular` | `"80"` | 895 | 0 |
| `both` | `"90"` | 746 | 18 |
| `both` | `"80"` | 1,326 | 32 |

The `strict` column is 0 for every `intracellular` row because those GPs have no source genes. The
permissive column is dominated by degenerate GPs: at precision `"80"` with `intercellular`, 431 GPs survive
permissively but only 32 still contain both partners, so 399 have been reduced to a single gene and no
longer encode an interaction. That is why section 2.3 uses the strict thresholds, and why this resource is
better suited to whole-transcriptome data than to a targeted panel — an intercellular GP is only two genes
wide and needs *both* partners probed.


In [ ]:
### Dataset ###
dataset = "xenium_human_breast_cancer"
species = "human"
batches = ["batch1", "batch2"]
spatial_key = "spatial"
n_neighbors = 8

### Prior GP mask (human PPI) ###
# See the parameter reference above for the full classification logic.

# Expected precision of the released predictions. Options:
#   "90" -> 17,849 interactions (conservative)
#   "80" -> 29,257 interactions (superset of "90", more false positives)
humanppi_precision = "80"

# Which interaction classes to keep, and how they are placed into GP components:
#   "intercellular" -> paracrine + juxtacrine; source = partner 1, target = partner 2
#   "intracellular" -> the rest; empty source, both partners in the target component
#   "both"          -> everything, each routed to the appropriate component
# NOTE: "intracellular" / "both" produce source-empty GPs, which section 2.3
# discards because it requires min_source_genes_per_gp=1.
humanppi_program_type = "intercellular"

# How to treat interactions whose partner locations stay unresolved after all
# available evidence. Options:
#   "exclude"       -> drop them (they cannot be classified either way)
#   "intracellular" -> keep them as intracellular
# How to treat proteins whose extracellular face is neither established nor
# contradicted (only a compatible keyword, and no topology available). Options:
#   "extracellular"  -> they count as able to act between cells
#   "intracellular"  -> they do not
# Has NO effect while humanppi_use_topology is True, since topology resolves
# every such protein either way. Only matters with use_topology = False.
humanppi_ambiguous_locality = "extracellular"

humanppi_unresolved_locality = "exclude"

# Reclassify interactions whose partners are subunits of a common protein
# complex (EBI Complex Portal plus curated gene families) as "cis_complex",
# since such complexes assemble within one cell. Only reclassifies; never adds
# interactions.
humanppi_detect_cis_complexes = True

# Drop interactions that cannot represent an interaction between two proteins:
#   immunoglobulin / T cell receptor V, D and J gene segments, which are
#   recombined into a single chain, and paralogue pairs that form no heterodimer
humanppi_filter_ig_tcr_segments = True
humanppi_filter_paralog_cross_pairs = True

# Retrieve membrane topology from UniProt and use it to decide whether a
# protein can present an extracellular face. Needs network access on first use
# and is cached afterwards.
humanppi_use_topology = True

# Shortest extracellular domain, in amino acids, that still counts as able to
# reach a partner on a neighboring cell. Contact-dependent interactions in
# which a partner protrudes less far are reclassified as "cis_complex", since
# they take place within one membrane (CD247 has a 9 aa ectodomain, KCNJ6 24).
# Set to 0 to disable this test. Requires humanppi_use_topology = True.
humanppi_min_extracellular_domain_length = 30

# Orient interactions from evidence about which partner sends the signal, so
# that the ligand ends up in the source (neighborhood) component and the
# receptor in the target (self) component. Options:
#   True  -> curated gene families, then the OmniPath intercell annotation,
#            then the GO molecular functions and the GPI anchor decide.
#            918 of 1,584 interactions are oriented from evidence, and of the
#            215 curated directions the rules decide, 215 are correct.
#   False -> keep the order of the two columns in the released predictions,
#            which is arbitrary: the ligand lands in the source component for
#            only 104 of 224 curated directions.
humanppi_orient_juxtacrine_gps = True

# Emit each contact-dependent interaction twice, once per orientation, since it
# operates in both directions between two neighboring cells. Options:
#   False -> one GP per interaction, arbitrary but reproducible orientation
#   True  -> two GPs per interaction (juxtacrine: 769 -> 1,538 at precision 90),
#            biologically faithful but creates pairs of highly correlated GPs
humanppi_symmetric_juxtacrine_gps = False

# Optional extra confidence thresholds on the prediction scores.
#   humanppi_min_rf_prob -> minimum RoseTTAFold2-PPI probability (column "RFprob")
#   humanppi_min_af_prob -> minimum AlphaFold2 probability (column "AFprob")
# None keeps the authors' own precision calibration (recommended).
humanppi_min_rf_prob = None
humanppi_min_af_prob = None

# Merge GPs sharing the same source gene into hub-centric GPs. Options:
#   False -> one GP per interaction, and the interaction class stays in the GP name
#   True  -> hub-centric GPs, but they are renamed "<GENE>_combined_GP", which
#            DISCARDS the interaction class and, on this small panel, retains
#            fewer usable GPs (25 vs 32)
combine_gps = False

### Model ###
# AnnData keys
counts_key = "counts"
adj_key = "spatial_connectivities"
cat_covariates_keys = ["batch"]
gp_names_key = "nichecompass_gp_names"
active_gp_names_key = "nichecompass_active_gp_names"
gp_targets_mask_key = "nichecompass_gp_targets"
gp_targets_categories_mask_key = "nichecompass_gp_targets_categories"
gp_sources_mask_key = "nichecompass_gp_sources"
gp_sources_categories_mask_key = "nichecompass_gp_sources_categories"
latent_key = "nichecompass_latent"

# Architecture
cat_covariates_embeds_injection = ["gene_expr_decoder"]
cat_covariates_embeds_nums = [2] # two samples
cat_covariates_no_edges = [True]
conv_layer_encoder = "gatv2conv" # change to "gcnconv" to save compute and memory
active_gp_thresh_ratio = 0.01

# Trainer
n_epochs = 400
n_epochs_all_gps = 25
lr = 0.001
lambda_edge_recon = 5000000.
lambda_gene_expr_recon = 3000.
lambda_l1_masked = 0. # prior GP regularization
lambda_l1_addon = 30. # de novo GP regularization
edge_batch_size = 512 # increase if more memory available or decrease to save memory
n_sampled_neighbors = 4
use_cuda_if_available = True

### Analysis ###
cell_type_key = "cell_states"
latent_leiden_resolution = 0.2
latent_cluster_key = f"latent_leiden_{str(latent_leiden_resolution)}"
sample_key = "batch"
spot_size = 30 # Xenium coordinates are in microns
differential_gp_test_results_key = "nichecompass_differential_gp_test_results"

### 1.3 Run Notebook Setup

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
# Get time of notebook execution for timestamping saved artifacts
now = datetime.now()
current_timestamp = now.strftime("%d%m%Y_%H%M%S")

### 1.4 Configure Paths

In [ ]:
# Define paths (relative to <repository_root>/analysis/data_analysis/)
ga_data_folder_path = "../../datasets/ga_data"
gp_data_folder_path = "../../datasets/gp_data"
so_data_folder_path = f"../../datasets/st_data/{dataset}"
humanppi_network_file_path = f"{gp_data_folder_path}/humanppi_network_{humanppi_precision}.csv"
omnipath_annotation_file_path = \
    f"{gp_data_folder_path}/omnipath_intercell_annotation.tsv"
complex_portal_file_path = f"{gp_data_folder_path}/complex_portal_human.tsv"
humanppi_topology_file_path = f"{gp_data_folder_path}/humanppi_protein_topology.tsv"
gene_orthologs_mapping_file_path = f"{ga_data_folder_path}/human_mouse_gene_orthologs.csv"
artifacts_folder_path = "../../artifacts"
model_folder_path = f"{artifacts_folder_path}/{dataset}_humanppi/{current_timestamp}/model"
figure_folder_path = f"{artifacts_folder_path}/{dataset}_humanppi/{current_timestamp}/figures"

### 1.5 Create Directories

In [ ]:
os.makedirs(model_folder_path, exist_ok=True)
os.makedirs(figure_folder_path, exist_ok=True)
os.makedirs(gp_data_folder_path, exist_ok=True)

## 2. Prepare Model Training

### 2.1 Create Prior Knowledge Gene Program (GP) Mask

- NicheCompass expects a prior GP mask as input, which it will use to make its latent feature space interpretable (through linear masked decoders).
- The user can provide a custom GP mask to NicheCompass based on the biological question of interest.
- In the other tutorials, the GP mask is built from OmniPath (Ligand-Receptor GPs), MEBOCOST (Enzyme-Sensor GPs) and NicheNet (Combined Interaction GPs).
- **Here we instead build the GP mask exclusively from predicted human protein-protein interactions** (PPI GPs), retrieved from the human interactome of Zhang, Humphreys et al. (Science, 2025).
- Every interaction is classified as **paracrine**, **juxtacrine**, **cis_complex**, **extracellular_assembly** or **intracellular** from the UniProt cellular-component keywords of its two partners, its membrane topology, the length of its extracellular domains, and protein-complex membership (see the parameter reference in section 1.2 for the full logic). Only paracrine and juxtacrine interactions become intercellular GPs, in which one partner is placed in the source (neighborhood) component and the other in the target (self) component, thereby modelling signalling between neighbouring cells.
- The interaction class is part of the GP name (```<gene 1>_<gene 2>_<class>_ppi_GP```) and the location class of each protein is used as its gene category, so both remain visible in the GP summaries in section 4.

In [ ]:
# Retrieve human PPI GPs (source: first interaction partner; target: second interaction partner)
humanppi_gp_dict = extract_gp_dict_from_humanppi_interactions(
    species=species,
    precision=humanppi_precision,
    program_type=humanppi_program_type,
    ambiguous_locality=humanppi_ambiguous_locality,
    unresolved_locality=humanppi_unresolved_locality,
    filter_ig_tcr_segments=humanppi_filter_ig_tcr_segments,
    filter_paralog_cross_pairs=humanppi_filter_paralog_cross_pairs,
    use_topology=humanppi_use_topology,
    topology_file_path=humanppi_topology_file_path,
    detect_cis_complexes=humanppi_detect_cis_complexes,
    min_extracellular_domain_length=humanppi_min_extracellular_domain_length,
    orient_juxtacrine_gps=humanppi_orient_juxtacrine_gps,
    omnipath_annotation_file_path=omnipath_annotation_file_path,
    symmetric_juxtacrine_gps=humanppi_symmetric_juxtacrine_gps,
    complex_portal_file_path=complex_portal_file_path,
    min_rf_prob=humanppi_min_rf_prob,
    min_af_prob=humanppi_min_af_prob,
    load_from_disk=os.path.exists(humanppi_network_file_path),
    save_to_disk=not os.path.exists(humanppi_network_file_path),
    ppi_network_file_path=humanppi_network_file_path,
    gene_orthologs_mapping_file_path=gene_orthologs_mapping_file_path,
    plot_gp_gene_count_distributions=True,
    gp_gene_count_distributions_save_path=f"{figure_folder_path}" \
                                           "/humanppi_gp_gene_count_distributions.svg")

print(f"Number of human PPI gene programs: {len(humanppi_gp_dict)}.")

In [ ]:
# Display example human PPI GP
humanppi_gp_names = list(humanppi_gp_dict.keys())
random.shuffle(humanppi_gp_names)
humanppi_gp_name = humanppi_gp_names[0]
print(f"{humanppi_gp_name}: {humanppi_gp_dict[humanppi_gp_name]}")

Next we filter and combine the GPs. ```filter_and_combine_gp_dict_gps_v2``` merges GPs that share
exactly the same source genes, which turns the one-interaction-per-GP dictionary into hub-centric GPs
(e.g. all interactions of ```PDCD1``` as a source become a single ```PDCD1_combined_GP``` containing all of
its partners in the target component).

For a small targeted panel this is a trade-off: it reduces the number of GPs, but each remaining GP contains
more probed genes and is therefore less noisy and more likely to be retained as an active GP. Set
```combine_gps = False``` in section 1.2 to keep one GP per individual interaction instead.

In [ ]:
# Filter and combine GPs
if combine_gps:
    gp_dicts = [humanppi_gp_dict]
    combined_gp_dict = filter_and_combine_gp_dict_gps_v2(
        gp_dicts,
        verbose=False)
else:
    combined_gp_dict = dict(humanppi_gp_dict)

print(f"Number of gene programs before filtering and combining: {len(humanppi_gp_dict)}.")
print(f"Number of gene programs after filtering and combining: {len(combined_gp_dict)}.")

### 2.2 Load Data & Compute Spatial Neighbor Graph

- NicheCompass expects a precomputed spatial adjacency matrix stored in 'adata.obsp[adj_key]'.
- The user can customize the spatial neighbor graph construction based on the biological question of interest.
- In the sample integration setting, we will compute a separate spatial adjacency matrix for each sample and combine them as disconnected components.
- The Xenium data stores cell centroids in ```adata.obs['x_centroid']``` and ```adata.obs['y_centroid']```. The cell below writes them into ```adata.obsm[spatial_key]``` if that entry is missing, and makes sure that raw counts are available in ```adata.layers[counts_key]```.

In [ ]:
adata_batch_list = []

for batch in batches:
    print(f"Processing batch {batch}...")
    print("Loading data...")
    adata_batch = sc.read_h5ad(
        f"{so_data_folder_path}/{dataset}_{batch}.h5ad")

    # Ensure spatial coordinates are stored in adata.obsm
    if spatial_key not in adata_batch.obsm:
        print(f"Creating adata.obsm['{spatial_key}'] from x_centroid / y_centroid...")
        adata_batch.obsm[spatial_key] = adata_batch.obs[
            ["x_centroid", "y_centroid"]].values.astype("float32")

    # Ensure raw counts are stored in adata.layers
    if counts_key not in adata_batch.layers:
        print(f"Creating adata.layers['{counts_key}'] from adata.X...")
        adata_batch.layers[counts_key] = adata_batch.X.copy()

    print("Computing spatial neighborhood graph...\n")
    # Compute (separate) spatial neighborhood graphs
    sq.gr.spatial_neighbors(adata_batch,
                            coord_type="generic",
                            spatial_key=spatial_key,
                            n_neighs=n_neighbors)

    # Make adjacency matrix symmetric
    adata_batch.obsp[adj_key] = (
        adata_batch.obsp[adj_key].maximum(
            adata_batch.obsp[adj_key].T))
    adata_batch_list.append(adata_batch)
adata = ad.concat(adata_batch_list, join="inner")

# Combine spatial neighborhood graphs as disconnected components
batch_connectivities = []
len_before_batch = 0
for i in range(len(adata_batch_list)):
    if i == 0: # first batch
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[0].shape[0],
            (adata.shape[0] -
            adata_batch_list[0].shape[0])))
        batch_connectivities.append(sp.hstack(
            (adata_batch_list[0].obsp[adj_key],
            after_batch_connectivities_extension)))
    elif i == (len(adata_batch_list) - 1): # last batch
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata.shape[0] -
            adata_batch_list[i].shape[0])))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key])))
    else: # middle batches
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0], len_before_batch))
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata.shape[0] -
            adata_batch_list[i].shape[0] -
            len_before_batch)))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key],
            after_batch_connectivities_extension)))
    len_before_batch += adata_batch_list[i].shape[0]
adata.obsp[adj_key] = sp.vstack(batch_connectivities)

### 2.3 Add GP Mask to Data

We require each retained GP to have at least one source gene and at least one target gene probed in the
dataset (```min_source_genes_per_gp=1```, ```min_target_genes_per_gp=1```). This is essential for PPI GPs on a
targeted panel: without it, a two-gene interaction GP would be retained even when only one of its two partners
is measured, yielding degenerate single-gene GPs that no longer encode an interaction.

In [ ]:
# Add the GP dictionary as binary masks to the adata
add_gps_from_gp_dict_to_adata(
    gp_dict=combined_gp_dict,
    adata=adata,
    gp_targets_mask_key=gp_targets_mask_key,
    gp_targets_categories_mask_key=gp_targets_categories_mask_key,
    gp_sources_mask_key=gp_sources_mask_key,
    gp_sources_categories_mask_key=gp_sources_categories_mask_key,
    gp_names_key=gp_names_key,
    min_genes_per_gp=2,
    min_source_genes_per_gp=1,
    min_target_genes_per_gp=1,
    max_genes_per_gp=None,
    max_source_genes_per_gp=None,
    max_target_genes_per_gp=None)

print(f"Number of GPs retained after masking against the {adata.n_vars} probed genes: "
      f"{len(adata.uns[gp_names_key])}.")

In [ ]:
# Inspect the retained GPs and how many of their genes are actually probed
gp_mask_summary_df = pd.DataFrame({
    "gp_name": adata.uns[gp_names_key],
    "n_probed_source_genes": adata.varm[gp_sources_mask_key].sum(axis=0),
    "n_probed_target_genes": adata.varm[gp_targets_mask_key].sum(axis=0)})
gp_mask_summary_df["n_probed_genes"] = (
    gp_mask_summary_df["n_probed_source_genes"] +
    gp_mask_summary_df["n_probed_target_genes"])
gp_mask_summary_df = gp_mask_summary_df.sort_values(
    by="n_probed_genes", ascending=False).reset_index(drop=True)
display(gp_mask_summary_df.head(20))

### 2.4 Explore Data

In [ ]:
cell_type_colors = create_new_color_dict(
    adata=adata,
    cat_key=cell_type_key)

In [ ]:
samples = adata.obs[sample_key].unique().tolist()

In [ ]:
for sample in samples:
    adata_batch = adata[adata.obs[sample_key] == sample]

    print(f"Summary of sample {sample}:")
    print(f"Number of nodes (observations): {adata_batch.layers[counts_key].shape[0]}")
    print(f"Number of node features (genes): {adata_batch.layers[counts_key].shape[1]}")

    # Visualize cell-level annotated data in physical space
    sc.pl.spatial(adata_batch,
                  color=cell_type_key,
                  palette=cell_type_colors,
                  spot_size=spot_size)

## 3. Train Model

### 3.1 Initialize, Train & Save Model

In [ ]:
# Initialize model
model = NicheCompass(adata,
                     counts_key=counts_key,
                     adj_key=adj_key,
                     cat_covariates_embeds_injection=cat_covariates_embeds_injection,
                     cat_covariates_keys=cat_covariates_keys,
                     cat_covariates_no_edges=cat_covariates_no_edges,
                     cat_covariates_embeds_nums=cat_covariates_embeds_nums,
                     gp_names_key=gp_names_key,
                     active_gp_names_key=active_gp_names_key,
                     gp_targets_mask_key=gp_targets_mask_key,
                     gp_targets_categories_mask_key=gp_targets_categories_mask_key,
                     gp_sources_mask_key=gp_sources_mask_key,
                     gp_sources_categories_mask_key=gp_sources_categories_mask_key,
                     latent_key=latent_key,
                     conv_layer_encoder=conv_layer_encoder,
                     active_gp_thresh_ratio=active_gp_thresh_ratio)

In [ ]:
# Train model
model.train(n_epochs=n_epochs,
            n_epochs_all_gps=n_epochs_all_gps,
            lr=lr,
            lambda_edge_recon=lambda_edge_recon,
            lambda_gene_expr_recon=lambda_gene_expr_recon,
            lambda_l1_masked=lambda_l1_masked,
            lambda_l1_addon=lambda_l1_addon,
            edge_batch_size=edge_batch_size,
            n_sampled_neighbors=n_sampled_neighbors,
            use_cuda_if_available=use_cuda_if_available,
            verbose=False)

In [ ]:
# Compute latent neighbor graph
sc.pp.neighbors(model.adata,
                use_rep=latent_key,
                key_added=latent_key)

# Compute UMAP embedding
sc.tl.umap(model.adata,
           neighbors_key=latent_key)

In [ ]:
# Save trained model
model.save(dir_path=model_folder_path,
           overwrite=True,
           save_adata=True,
           adata_file_name="adata.h5ad")

## 4. Analysis

In [ ]:
load_timestamp = current_timestamp
# load_timestamp = "<timestamp>" # set this to load a previously trained model

figure_folder_path = f"{artifacts_folder_path}/{dataset}_humanppi/{load_timestamp}/figures"
model_folder_path = f"{artifacts_folder_path}/{dataset}_humanppi/{load_timestamp}/model"

os.makedirs(figure_folder_path, exist_ok=True)

In [ ]:
# Load trained model
model = NicheCompass.load(dir_path=model_folder_path,
                          adata=None,
                          adata_file_name="adata.h5ad",
                          gp_names_key=gp_names_key)

In [ ]:
samples = model.adata.obs[sample_key].unique().tolist()

### 4.1 Visualize NicheCompass Latent GP Space

Let's inspect how well the integration worked by visualizing the batch annotations in the latent GP space.

In [ ]:
batch_colors = create_new_color_dict(
    adata=model.adata,
    cat_key=cat_covariates_keys[0])

In [ ]:
cell_type_colors = create_new_color_dict(
    adata=model.adata,
    cat_key=cell_type_key)

In [ ]:
# Create plot of batch annotations in physical and latent space
groups = None
save_fig = True
file_path = f"{figure_folder_path}/" \
            "batches_latent_physical_space.svg"

fig = plt.figure(figsize=(12, 14))
title = fig.suptitle(t=f"NicheCompass Batches " \
                       "in Latent and Physical Space",
                     y=0.96,
                     x=0.55,
                     fontsize=20)
spec1 = gridspec.GridSpec(ncols=1,
                          nrows=2,
                          width_ratios=[1],
                          height_ratios=[3, 2])
spec2 = gridspec.GridSpec(ncols=len(samples),
                          nrows=2,
                          width_ratios=[1] * len(samples),
                          height_ratios=[3, 2])
axs = []
axs.append(fig.add_subplot(spec1[0]))
sc.pl.umap(adata=model.adata,
           color=[cat_covariates_keys[0]],
           groups=groups,
           palette=batch_colors,
           title=f"Batches in Latent Space",
           ax=axs[0],
           show=False)
for idx, sample in enumerate(samples):
    axs.append(fig.add_subplot(spec2[len(samples) + idx]))
    sc.pl.spatial(adata=model.adata[model.adata.obs[sample_key] == sample],
                  color=[cat_covariates_keys[0]],
                  groups=groups,
                  palette=batch_colors,
                  spot_size=spot_size,
                  title=f"Batches in Physical Space \n"
                        f"(Sample: {sample})",
                  legend_loc=None,
                  ax=axs[idx+1],
                  show=False)

# Create and position shared legend
handles, labels = axs[0].get_legend_handles_labels()
lgd = fig.legend(handles,
                 labels,
                 loc="center left",
                 bbox_to_anchor=(0.98, 0.5))
axs[0].get_legend().remove()

# Adjust, save and display plot
plt.subplots_adjust(wspace=0.2, hspace=0.25)
if save_fig:
    fig.savefig(file_path,
                bbox_extra_artists=(lgd, title),
                bbox_inches="tight")
plt.show()

Next, let's look at the preservation of cell state annotations in the latent GP space. Note that the goal of NicheCompass is not a separation of cell types but rather to identify spatially consistent cell niches.

In [ ]:
# Create plot of cell state annotations in physical and latent space
groups = None
save_fig = True
file_path = f"{figure_folder_path}/" \
            "cell_types_latent_physical_space.svg"

fig = plt.figure(figsize=(12, 14))
title = fig.suptitle(t=f"NicheCompass Cell States " \
                       "in Latent and Physical Space",
                     y=0.96,
                     x=0.55,
                     fontsize=20)
spec1 = gridspec.GridSpec(ncols=1,
                          nrows=2,
                          width_ratios=[1],
                          height_ratios=[3, 2])
spec2 = gridspec.GridSpec(ncols=len(samples),
                          nrows=2,
                          width_ratios=[1] * len(samples),
                          height_ratios=[3, 2])
axs = []
axs.append(fig.add_subplot(spec1[0]))
sc.pl.umap(adata=model.adata,
           color=[cell_type_key],
           groups=groups,
           palette=cell_type_colors,
           title=f"Cell States in Latent Space",
           ax=axs[0],
           show=False)
for idx, sample in enumerate(samples):
    axs.append(fig.add_subplot(spec2[len(samples) + idx]))
    sc.pl.spatial(adata=model.adata[model.adata.obs[sample_key] == sample],
                  color=[cell_type_key],
                  groups=groups,
                  palette=cell_type_colors,
                  spot_size=spot_size,
                  title=f"Cell States in Physical Space \n"
                        f"(Sample: {sample})",
                  legend_loc=None,
                  ax=axs[idx+1],
                  show=False)

# Create and position shared legend
handles, labels = axs[0].get_legend_handles_labels()
lgd = fig.legend(handles,
                 labels,
                 loc="center left",
                 bbox_to_anchor=(0.98, 0.5))
axs[0].get_legend().remove()

# Adjust, save and display plot
plt.subplots_adjust(wspace=0.2, hspace=0.25)
if save_fig:
    fig.savefig(file_path,
                bbox_extra_artists=(lgd, title),
                bbox_inches="tight")
plt.show()

### 4.2 Identify Niches

We compute Leiden clustering of the NicheCompass latent GP space to identify spatially consistent cell niches.

In [ ]:
# Compute latent Leiden clustering
sc.tl.leiden(adata=model.adata,
             resolution=latent_leiden_resolution,
             key_added=latent_cluster_key,
             neighbors_key=latent_key)

In [ ]:
latent_cluster_colors = create_new_color_dict(
    adata=model.adata,
    cat_key=latent_cluster_key)

In [ ]:
# Create plot of latent cluster / niche annotations in physical and latent space
groups = None
save_fig = True
file_path = f"{figure_folder_path}/" \
            "res_{latent_leiden_resolution}_niches_latent_physical_space.svg"

fig = plt.figure(figsize=(12, 14))
title = fig.suptitle(t=f"NicheCompass Niches " \
                       "in Latent and Physical Space",
                     y=0.96,
                     x=0.55,
                     fontsize=20)
spec1 = gridspec.GridSpec(ncols=1,
                          nrows=2,
                          width_ratios=[1],
                          height_ratios=[3, 2])
spec2 = gridspec.GridSpec(ncols=len(samples),
                          nrows=2,
                          width_ratios=[1] * len(samples),
                          height_ratios=[3, 2])
axs = []
axs.append(fig.add_subplot(spec1[0]))
sc.pl.umap(adata=model.adata,
           color=[latent_cluster_key],
           groups=groups,
           palette=latent_cluster_colors,
           title=f"Niches in Latent Space",
           ax=axs[0],
           show=False)
for idx, sample in enumerate(samples):
    axs.append(fig.add_subplot(spec2[len(samples) + idx]))
    sc.pl.spatial(adata=model.adata[model.adata.obs[sample_key] == sample],
                  color=[latent_cluster_key],
                  groups=groups,
                  palette=latent_cluster_colors,
                  spot_size=spot_size,
                  title=f"Niches in Physical Space \n"
                        f"(Sample: {sample})",
                  legend_loc=None,
                  ax=axs[idx+1],
                  show=False)

# Create and position shared legend
handles, labels = axs[0].get_legend_handles_labels()
lgd = fig.legend(handles,
                 labels,
                 loc="center left",
                 bbox_to_anchor=(0.98, 0.5))
axs[0].get_legend().remove()

# Adjust, save and display plot
plt.subplots_adjust(wspace=0.2, hspace=0.25)
if save_fig:
    fig.savefig(file_path,
                bbox_extra_artists=(lgd, title),
                bbox_inches="tight")
plt.show()

### 4.3 Characterize Niches

Now we will characterize the identified cell niches.

#### 4.3.1 Niche Composition

We can analyze the niche composition in terms of batch and cell state labels.

In [ ]:
save_fig = True
file_path = f"{figure_folder_path}/" \
            f"res_{latent_leiden_resolution}_" \
            f"niche_composition_batches.svg"

df_counts = (model.adata.obs.groupby([latent_cluster_key, cat_covariates_keys[0]])
             .size().unstack())
df_counts.plot(kind="bar", stacked=True, figsize=(10,10))
legend = plt.legend(bbox_to_anchor=(1, 1), loc="upper left", prop={'size': 10})
legend.set_title("Batch Annotations", prop={'size': 10})
plt.title("Batch Composition of Niches")
plt.xlabel("Niche")
plt.ylabel("Cell Counts")
if save_fig:
    plt.savefig(file_path,
                bbox_extra_artists=(legend,),
                bbox_inches="tight")

In [ ]:
save_fig = True
file_path = f"{figure_folder_path}/" \
            f"res_{latent_leiden_resolution}_" \
            f"niche_composition_cell_types.svg"

df_counts = (model.adata.obs.groupby([latent_cluster_key, cell_type_key])
             .size().unstack())
df_counts.plot(kind="bar", stacked=True, figsize=(10,10))
legend = plt.legend(bbox_to_anchor=(1, 1), loc="upper left", prop={'size': 10})
legend.set_title("Cell State Annotations", prop={'size': 10})
plt.title("Cell State Composition of Niches")
plt.xlabel("Niche")
plt.ylabel("Cell Counts")
if save_fig:
    plt.savefig(file_path,
                bbox_extra_artists=(legend,),
                bbox_inches="tight")

#### 4.3.2 Differential GPs

Now we can test which GPs are differentially expressed in a niche. To this end, we will perform "one-vs-rest" differential GP testing, i.e all niches (```selected_cats = None```) are tested against all other niches (```comparison_cats = "rest"```). However, differential GP testing can also be performed in the following ways:
- Set ```selected_cats = ["0"]``` to perform differential GP testing for a specific niche only, in this case niche "0".
- Set ```comparison_cats = ["2"]``` to perform differential GP testing against niche "2" as opposed to against all other niches.

We choose an absolute log bayes factor threshold of 2.3 to determine strongly enriched GPs (see https://en.wikipedia.org/wiki/Bayes_factor).

Note that with a small prior GP mask, a substantial part of the latent space consists of de novo GPs. Prior PPI GPs are named ```<GENE1>_<GENE2>_<class>_ppi_GP``` with the class being ```paracrine``` or ```juxtacrine```, which makes them easy to distinguish from de novo GPs and shows the interaction class directly in the results below.

In [ ]:
# Check number of active GPs
active_gps = model.get_active_gps()
print(f"Number of total gene programs: {len(model.adata.uns[gp_names_key])}.")
print(f"Number of active gene programs: {len(active_gps)}.")

In [ ]:
# Split active GPs into prior PPI GPs and de novo GPs
prior_active_gps = [gp for gp in active_gps if gp.endswith("_ppi_GP")
                    or gp.endswith("_combined_GP")]
de_novo_active_gps = [gp for gp in active_gps if gp not in prior_active_gps]
print(f"Active prior human PPI GPs: {len(prior_active_gps)}.")
print(f"Active de novo GPs: {len(de_novo_active_gps)}.")
print(f"\nActive prior human PPI GPs:\n{prior_active_gps}")

In [ ]:
# Display example active GPs
gp_summary_df = model.get_gp_summary()
gp_summary_df[gp_summary_df["gp_active"] == True].head()

In [ ]:
# Set parameters for differential gp testing
selected_cats = None
comparison_cats = "rest"
title = f"NicheCompass Strongly Enriched Niche GPs"
log_bayes_factor_thresh = 2.3
save_fig = True
file_path = f"{figure_folder_path}/" \
            f"/log_bayes_factor_{log_bayes_factor_thresh}" \
             "_niches_enriched_gps_heatmap.svg"

In [ ]:
# Run differential gp testing
enriched_gps = model.run_differential_gp_tests(
    cat_key=latent_cluster_key,
    selected_cats=selected_cats,
    comparison_cats=comparison_cats,
    log_bayes_factor_thresh=log_bayes_factor_thresh)

In [ ]:
# Results are stored in a df in the adata object
model.adata.uns[differential_gp_test_results_key]

In [ ]:
# Visualize GP activities of enriched GPs across niches
df = model.adata.obs[[latent_cluster_key] + enriched_gps].groupby(latent_cluster_key).mean()

scaler = MinMaxScaler()
normalized_columns = scaler.fit_transform(df)
normalized_df = pd.DataFrame(normalized_columns, columns=df.columns)
normalized_df.index = df.index

plt.figure(figsize=(16, 8))  # Set the figure size
ax = sns.heatmap(normalized_df,
            cmap='viridis',
            annot=False,
            linewidths=0)
plt.xticks(rotation=45,
           fontsize=8,
           ha="right"
          )
plt.xlabel("Gene Programs", fontsize=16)
plt.savefig(f"{figure_folder_path}/enriched_gps_heatmap.svg",
            bbox_inches="tight")

In [ ]:
# Store gene program summary of enriched gene programs
save_file = True
file_path = f"{figure_folder_path}/" \
            f"/log_bayes_factor_{log_bayes_factor_thresh}_" \
            "niche_enriched_gps_summary.csv"

gp_summary_cols = ["gp_name",
                   "n_source_genes",
                   "n_non_zero_source_genes",
                   "n_target_genes",
                   "n_non_zero_target_genes",
                   "gp_source_genes",
                   "gp_target_genes",
                   "gp_source_genes_importances",
                   "gp_target_genes_importances"]

enriched_gp_summary_df = gp_summary_df[gp_summary_df["gp_name"].isin(enriched_gps)]
cat_dtype = pd.CategoricalDtype(categories=enriched_gps, ordered=True)
enriched_gp_summary_df.loc[:, "gp_name"] = enriched_gp_summary_df["gp_name"].astype(cat_dtype)
enriched_gp_summary_df = enriched_gp_summary_df.sort_values(by="gp_name")
enriched_gp_summary_df = enriched_gp_summary_df[gp_summary_cols]

if save_file:
    enriched_gp_summary_df.to_csv(f"{file_path}")
else:
    display(enriched_gp_summary_df)

Now we will have a look at the GP activities and the log normalized counts of
the most important omics features of the differential GPs.

In [ ]:
plot_label = f"log_bayes_factor_{log_bayes_factor_thresh}_cluster_{selected_cats[0] if selected_cats else 'None'}_vs_rest"
save_figs = True

generate_enriched_gp_info_plots(
    plot_label=plot_label,
    model=model,
    sample_key=sample_key,
    differential_gp_test_results_key=differential_gp_test_results_key,
    cat_key=latent_cluster_key,
    cat_palette=latent_cluster_colors,
    n_top_enriched_gp_start_idx=0,
    n_top_enriched_gp_end_idx=10,
    feature_spaces=samples, # ["latent"]
    n_top_genes_per_gp=3,
    save_figs=save_figs,
    figure_folder_path=f"{figure_folder_path}/",
    spot_size=spot_size)

#### 4.3.3 Cell-cell Communication

Now we will use the inferred activity of an enriched human PPI GP to analyze the involved intercellular
interactions. We pick the most strongly enriched prior PPI GP so that this cell works regardless of which GPs
were retained for your gene panel; set ```gp_name``` manually to inspect a specific interaction (for example
```"PDCD1_combined_GP"``` for the PD-1 / PD-L1 axis).

In [ ]:
# Select the most strongly enriched prior human PPI GP
enriched_prior_gps = [gp for gp in enriched_gps if gp.endswith("_ppi_GP")
                      or gp.endswith("_combined_GP")]
print(f"Enriched prior human PPI GPs:\n{enriched_prior_gps}\n")

gp_name = enriched_prior_gps[0] if enriched_prior_gps else enriched_gps[0]
print(f"Selected GP: {gp_name}")

In [ ]:
network_df = compute_communication_gp_network(
    gp_list=[gp_name],
    model=model,
    group_key=latent_cluster_key,
    n_neighbors=n_neighbors)

visualize_communication_gp_network(
    adata=model.adata,
    network_df=network_df,
    figsize=(9, 7),
    cat_colors=latent_cluster_colors,
    edge_type_colors=["#1f77b4"],
    cat_key=latent_cluster_key,
    save=True,
    save_path=f"{figure_folder_path}/gp_network_{gp_name}.svg",
    )